# Blocked vs Interleaved — accuracy comparison

Conditions: `blocked` (birds.md), `interleaved` (birds_interleaved.md).  
Answer key: Q01=A, Q02=C, Q03=B, Q04=A, Q05=C, Q06=C, Q07=A, Q08=B, Q09=C, Q10=B

In [ ]:
import csv
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

HERE = Path.cwd()
BLOCKED_PATH    = HERE / 'blocked.csv'
INTERLEAVED_PATH = HERE / 'interleaved.csv'

QIDS    = [f'Q{n:02d}' for n in range(1, 11)]
LETTERS = ['A', 'B', 'C', 'D', 'E']
CONDS   = ['blocked', 'interleaved']
COLORS  = {'blocked': '#1f77b4', 'interleaved': '#d62728'}

ATTN_CHECKS = {'AT1': 'Gold versus green', 'AT2': 'Plumage strategy'}

In [ ]:
def load(path):
    with open(path, newline='', encoding='utf-8-sig') as f:
        rows = list(csv.DictReader(f))
    return [r for r in rows[2:] if r.get('Finished', '').lower() in {'true', '1'}]

def passed_attention(r):
    return all(r.get(k, '').strip() == v for k, v in ATTN_CHECKS.items())

raw   = {'blocked': load(BLOCKED_PATH), 'interleaved': load(INTERLEAVED_PATH)}
data  = {c: [r for r in raw[c] if  passed_attention(r)] for c in CONDS}
failed = {c: [r for r in raw[c] if not passed_attention(r)] for c in CONDS}

for c in CONDS:
    print(f'{c}: {len(raw[c])} finished, {len(failed[c])} failed attention, {len(data[c])} retained')

In [ ]:
# Answer key: Q01–Q10
ANSWERS = dict(zip(QIDS, ['A', 'C', 'B', 'A', 'C', 'C', 'A', 'B', 'C', 'B']))

def resp_set(r, qid):
    """Return frozenset of selected letters (handles comma-separated multi-select)."""
    val = r.get(qid, '').strip()
    if not val:
        return frozenset()
    return frozenset(p.strip() for p in val.split(',') if p.strip() in LETTERS)

def is_correct(r, qid):
    return resp_set(r, qid) == frozenset([ANSWERS[qid]])

def correctness_matrix(rows):
    return np.array([[int(is_correct(r, qid)) for qid in QIDS] for r in rows])

Ms = {c: correctness_matrix(data[c]) for c in CONDS}

## SC0 score distribution

In [ ]:
RNG = np.random.default_rng(42)
N_BOOT = 2000

def bootstrap_ci(arr, ci=95):
    if len(arr) == 0:
        return (0.0, 0.0)
    boots = RNG.choice(arr, size=(N_BOOT, len(arr)), replace=True).mean(axis=1)
    return np.percentile(boots, (100 - ci) / 2), np.percentile(boots, 100 - (100 - ci) / 2)

# Computed accuracy scores (0–10)
computed_sc = {c: Ms[c].sum(axis=1).tolist() for c in CONDS}

# SC0-based scores (subtract AT contributions) for cross-check
def adj_score(r):
    sc0 = int(r.get('SC0', 0) or 0)
    return sc0 - sum(1 for k, v in ATTN_CHECKS.items() if r.get(k, '').strip() == v)

sc0_adj = {c: [adj_score(r) for r in data[c]] for c in CONDS}

print(f'{"Condition":<14} {"n":>4}  {"computed mean":>14}  {"SC0-adj mean":>12}')
for c in CONDS:
    cm = np.mean(computed_sc[c]); sm = np.mean(sc0_adj[c])
    print(f'{c:<14} {len(computed_sc[c]):>4}  {cm:>14.2f}  {sm:>12.2f}')

bins = np.arange(-0.5, 11.5, 1)

fig, ax = plt.subplots(figsize=(10, 4))
for c in CONDS:
    s = computed_sc[c]
    ax.hist(s, bins=bins, alpha=0.55, color=COLORS[c],
            label=f'{c} (n={len(s)}, mean={np.mean(s):.1f})',
            edgecolor='white', linewidth=0.5)
    ax.axvline(np.mean(s), color=COLORS[c], ls='--', lw=1.5)

ax.set_xticks(np.arange(0, 11))
ax.set_xlabel('Score (correct out of 10)')
ax.set_ylabel('Count')
ax.set_title('Score distribution: blocked vs interleaved')
ax.legend()
plt.tight_layout()
fig.savefig('score_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Per-question accuracy (with bootstrap CIs)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))
x = np.arange(len(QIDS))
w = 0.38
for k, c in enumerate(CONDS):
    M = Ms[c]
    accs = M.mean(axis=0)
    errs_lo, errs_hi = [], []
    for j in range(len(QIDS)):
        lo, hi = bootstrap_ci(M[:, j])
        errs_lo.append(accs[j] - lo)
        errs_hi.append(hi - accs[j])
    xpos = x + (k - 0.5) * w
    ax.bar(xpos, accs, w, color=COLORS[c], alpha=0.85,
           label=f'{c} (n={len(data[c])})')
    ax.errorbar(xpos, accs, yerr=[errs_lo, errs_hi],
                fmt='none', color='black', capsize=3, linewidth=1)
    for i, a in enumerate(accs):
        ax.text(xpos[i], a + errs_hi[i] + 0.03, f'{a:.0%}', ha='center', fontsize=7)

ax.axhline(0.20, color='red', ls='--', label='chance (1/5)')
ax.set_xticks(x)
ax.set_xticklabels([f'{q}\n({ANSWERS[q]})' for q in QIDS])
ax.set_ylim(0, 1.15)
ax.set_ylabel('Accuracy')
ax.set_xlabel('Question (correct answer shown)')
ax.set_title('Per-question accuracy: blocked vs interleaved (95% bootstrap CI)')
ax.legend()
plt.tight_layout()
fig.savefig('per_question_accuracy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Per-question answer distributions (correct answer outlined in black)

In [ ]:
def tally(rows, qid):
    counts = Counter({L: 0 for L in LETTERS})
    for r in rows:
        for part in r.get(qid, '').strip().split(','):
            part = part.strip()
            if part in LETTERS:
                counts[part] += 1
    return counts

fig, axes = plt.subplots(5, 2, figsize=(14, 22))
axes = axes.flatten()
x = np.arange(len(LETTERS))
w = 0.38

for ax, qid in zip(axes, QIDS):
    correct = ANSWERS[qid]
    correct_idx = LETTERS.index(correct)
    n_max = max(len(data[c]) for c in CONDS)

    # Shaded background behind correct answer column
    ax.axvspan(correct_idx - 0.5, correct_idx + 0.5,
               color='gold', alpha=0.25, zorder=0)

    for k, c in enumerate(CONDS):
        counts = tally(data[c], qid)
        vals = [counts[L] for L in LETTERS]
        ax.bar(x + (k - 0.5) * w, vals, w, color=COLORS[c], alpha=0.85,
               label=c if qid == QIDS[0] else None, zorder=2)
        for i, v in enumerate(vals):
            if v > 0:
                ax.text(x[i] + (k - 0.5) * w, v + 0.05, str(v),
                        ha='center', va='bottom', fontsize=7, zorder=3)

    ax.set_xticks(x)
    ax.set_xticklabels(LETTERS)
    ax.set_ylim(0, n_max + 1)
    ax.set_ylabel('Count')
    ax.set_title(f'{qid} (correct: {correct})', fontsize=9, loc='left')
    ax.tick_params(labelsize=8)

axes[0].legend(loc='upper right', fontsize=8)
plt.suptitle('Answer distributions: blocked vs interleaved (gold = correct answer)', y=1.01, fontsize=11)
plt.tight_layout()
fig.savefig('answer_distributions_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## SC0 scores including attention-check failures (all finished)

In [ ]:
for c in CONDS:
    print(f'\n{c}:')
    for r in raw[c]:
        pid = (r.get('PROLIFIC_PID') or '?')[:10]
        sc0 = int(r.get('SC0', 0) or 0)
        at_score = sum(1 for k, v in ATTN_CHECKS.items() if r.get(k, '').strip() == v)
        adj = sc0 - at_score
        attn = 'PASS' if passed_attention(r) else 'FAIL-ATTN (excluded)'
        print(f'  {pid}  SC0={sc0:>3}  adj={adj:>2}/10  {attn}')